# 03 — Interface & Explanation (v3, DAD)

How a plain-English question becomes a **verified** answer, and how the optional local LLM rewords it
**without ever changing a number**. Sections:
1. The deterministic parser (intents + scope)
2. Routing to the truth layer + trust labels
3. Charts attached to answers
4. The LLM explanation layer
5. The faithfulness gate (the core guarantee)
6. Putting it together (live)

Run **Restart & Run All** (after `bash tools/get_dad_dataset.sh`).

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import preprocess, interface, llm
from src import interface_parsing as ip
df = preprocess.load_week()
bot = interface.Interface(df)
print("interface ready")

interface ready


## 1. The deterministic parser (`interface_parsing`)

Routing is **keyword/regex only** — no model — so the *right query* is chosen deterministically. It
resolves an **intent**, an optional **day**, an **attack type**, and (for temperature) a **statistic**.
Being deterministic, it *refuses* rather than guesses when it can't map a question.

In [2]:
for q in ["how many attacks on wednesday?", "which ip is the attacker?",
          "show all the sensors", "show me the anomalies",
          "highest temperature on friday?", "busiest hour?", "nonsense words here"]:
    print(f"{q:38s} → intent={ip.detect_intent(q)}, day={ip.resolve_day(q)}, "
          f"type={ip.resolve_attack_type(q)}, stat={ip.resolve_stat(q)}")

how many attacks on wednesday?         → intent=count, day=wed, type=None, stat=mean
which ip is the attacker?              → intent=attacker, day=None, type=None, stat=mean
show all the sensors                   → intent=sensors, day=None, type=None, stat=mean
show me the anomalies                  → intent=anomalies, day=None, type=None, stat=mean
highest temperature on friday?         → intent=temperature, day=fri, type=None, stat=max
busiest hour?                          → intent=traffic, day=None, type=None, stat=mean
nonsense words here                    → intent=None, day=None, type=None, stat=mean


## 2. Routing to the truth layer + trust labels

`Interface.ask` calls `query_engine` (the exact numbers) and returns an `Answer` with **trust labels**:
🔢 *computed* (the exact figure), 📚 *context* (background), 🧠 *inferred* (a cautious interpretation).
The number is exact **before** any LLM is involved.

In [3]:
for q in ["How many attacks on Wednesday?", "Which IP is the attacker?",
          "Show all the sensors in the dataset", "Show me the anomalies",
          "What types of attack are in the data?", "Give me a summary of Thursday"]:
    a = bot.ask(q)
    print("Q:", q)
    print("   🔢", a.text)
    if a.context:  print("   📚", a.context)
    if a.inferred: print("   🧠", a.inferred)
    print()

Q: How many attacks on Wednesday?
   🔢 There were 104 labelled attack packets on Wednesday.

Q: Which IP is the attacker?
   🔢 The attack traffic comes from 10.6.56.41.
   📚 In DAD all attacks originate from a single injected host.

Q: Show all the sensors in the dataset
   🔢 The dataset has 16 sensors (MQTT client ids): [131, 132, 133, 134, 151, 152, 153, 154, 231, 232, 233, 234, 251, 252, 253, 254].
   📚 Client ids encode the cooling unit and probe — e.g. 131 = InRow 13, probe 1 (4 InRow units × 4 probes: supply-air, return-air, entering-fluid, leaving-fluid).

Q: Show me the anomalies
   🔢 There are 544 labelled anomalies (network attacks) across 4 days: Wednesday, Thursday, Friday, Saturday.
   📚 By type — duplication 56, mixed 384, modification 104.
   🧠 Anomalies here are attacks on the MQTT messages (duplication / interception / modification), not just temperature spikes.

Q: What types of attack are in the data?
   🔢 Labelled attack packets by type — duplication: 56, mixed: 384

### Unknown questions are refused, not faked
This is a feature, not a bug: a deterministic router will never invent an intent, so it never fabricates
an answer.

In [4]:
a = bot.ask("write me a poem about routers")
print("intent:", a.intent, "| text:", a.text)

intent: None | text: I couldn't map that to a known query. Try asking about attacks, attack types, the attacker IP, temperatures, or traffic.


## 3. Charts attached to answers
For the web UI, the interface attaches a small **presentation-only** chart payload (a bar of
attacks-per-day / per-sensor, or a line of temperature/traffic). The chart only *draws* numbers the
truth layer already computed — it can never introduce a new figure.

In [5]:
a = bot.ask("Show me the anomalies")
print("chart kind:", a.chart["kind"], "| x:", a.chart["x"], "| y:", a.chart["y"])
b = bot.ask("Show all the sensors in the dataset")
print("chart kind:", b.chart["kind"], "| first 4 sensors:", b.chart["x"][:4], "counts:", b.chart["y"][:4])

chart kind: bar | x: ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'] | y: [0, 0, 104, 56, 192, 192, 0]
chart kind: bar | first 4 sensors: ['131', '132', '133', '134'] counts: [2021, 2021, 2021, 2021]


## 4. The LLM explanation layer (`llm`)
When a local **Ollama** model is running, `Interface(use_llm=True)` asks it to reword the computed facts
into friendlier prose. The model is given the facts verbatim and instructed never to change a number;
it is a *re-phraser*, not a calculator.

In [6]:
print("Ollama available:", llm.is_available())
print("pulled models   :", llm.list_models()[:3])

Ollama available: True
pulled models   : ['llama3.2:3b', 'hf.co/Andycurrent/Gemma-3-1B-it-GLM-4.7-Flash-Heretic-Uncensored-Thinking_GGUF:F16', 'hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q6_K_P']


## 5. The faithfulness gate — the core guarantee
After the LLM rewords, `llm.faithfulness` extracts the numbers from the facts and from the prose and
checks that the prose introduced **no new figure**. Rounding is allowed; clock times are ignored; IP
octets and counts are guarded. If anything was invented, the interface **discards the prose and returns
the exact template**.

In [7]:
facts = "There were 104 labelled attack packets on Wednesday; the attacker is 10.6.56.41."
print("faithful rewording:", llm.faithfulness(facts, "On Wednesday, host 10.6.56.41 sent 104 attack packets."))
print("fabricated count  :", llm.faithfulness(facts, "There were about 900 attacks on Wednesday."))
print("rounding allowed  :", llm.faithfulness("max temp 29.28 °C", "the peak was about 29.3 °C"))

faithful rewording: {'ok': True, 'dropped': [], 'invented': []}
fabricated count  : {'ok': False, 'dropped': [10.6, 56.41, 104.0], 'invented': [900.0]}
rounding allowed  : {'ok': True, 'dropped': [], 'invented': []}


## 6. Putting it together (live)
With the gate in place, turning the LLM on is safe. If Ollama is running you'll see the reworded prose
(with `used_llm=True`); if not, the exact template is shown — either way the numbers are guaranteed.

In [8]:
q = "How many attacks on Wednesday?"
print("template :", interface.Interface(df, use_llm=False).ask(q).text)
if llm.is_available():
    a = interface.Interface(df, use_llm=True).ask(q)
    print("AI       :", a.text, "| used_llm =", a.used_llm)
else:
    print("AI       : (no local Ollama running → exact template used)")

template : There were 104 labelled attack packets on Wednesday.


AI       : On Wednesday, there were 104 labelled attack packets. | used_llm = True


## Findings
The interface gives a natural, chart-augmented question-answering experience while keeping the project's
promise **structurally**: deterministic routing → exact query engine → trust-labelled answer → optional
LLM wording → faithfulness gate. A cheap local model can be used safely because the numbers are exact
upstream and re-checked downstream. See `report/REPORT.md` §5.3 and the web UI (`apps/web/`).